# M4d — Verificação manual dos resultados no conjunto de teste holdout

Carrega o melhor modelo treinado (notebook 06) e roda a inferência sobre os documentos extraídos pelo notebook 08 (`data/documento_social/teste_holdout/extracao/`) — documentos que **não** entraram no treino nem no split de validação usados nos notebooks 01-06. Aplica o mesmo pós-processamento heurístico (`scripts/pos_processamento_ner.py`) usado na avaliação quantitativa do notebook 10.

**Inspeção puramente visual/qualitativa**: mostra o que o modelo previu (texto, tipo, página) e desenha as caixas sobre o PDF, sem carregar gabarito — é o passo final do fluxo, pra você olhar documento por documento e formar opinião própria sobre a qualidade das previsões. Para números de precision/recall/F1 de verdade (com gabarito gerado nos notebooks 07 e 09), ver `notebooks/10.extrair_metricas_holdout.ipynb`.

**Pré-requisito:** rodar `notebooks/08.extrair_texto_holdout.ipynb` antes (kernel `locscan-ocr`), com os documentos já colocados em `data/documento_social/teste_holdout/arquivos/`. Este notebook roda no kernel GERAL do projeto (`.venv`, torch/transformers).

In [1]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from transformers import AutoModelForTokenClassification, AutoTokenizer

print(f"CUDA disponível: {torch.cuda.is_available()}")

d:\Arquivos\ProjetosPython\BFC\LocalScanDocument_v3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA disponível: True


## Configuração

`INCLUIR_SINTETICOS` escolhe qual experimento usar (deve bater com o que você quer testar — ver notebook 06). `CENARIO_MANUAL` força um cenário específico em vez do melhor por F1.

In [2]:
def encontrar_raiz_projeto(marcador: str = "requirements.txt") -> Path:
    p = Path.cwd()
    for _ in range(6):
        if (p / marcador).exists():
            return p
        p = p.parent
    raise RuntimeError(f"Não encontrei a raiz do projeto ({marcador}) subindo a partir de {Path.cwd()}")

RAIZ = encontrar_raiz_projeto()
sys.path.insert(0, str(RAIZ / "scripts"))
from pos_processamento_ner import extrair_nome_assinatura_digital, propagar_entidades_por_texto_identico

INCLUIR_SINTETICOS = False
CENARIO_MANUAL = None  # ex: "c07_weight_decay_alto" -- None usa o melhor por F1 do experimento
APLICAR_REGEX_ASSINATURA_DIGITAL = True  # ver anotacoes.txt 2026-09-11 -- ganho real, pequeno
APLICAR_PROPAGACAO_TEXTO_IDENTICO = True  # repropaga CPF/RG/NOME/ENDERECO já visto em outra página do mesmo doc (ver conversa 2026-09-11)

MAX_LENGTH = 512

PASTA_EXTRACAO_HOLDOUT = RAIZ / "data" / "documento_social" / "teste_holdout" / "extracao"
PASTA_ARQUIVOS_HOLDOUT = RAIZ / "data" / "documento_social" / "teste_holdout" / "arquivos"

_sufixo_experimento = "-com-sinteticos" if INCLUIR_SINTETICOS else "-sem-sinteticos"
PASTA_CENARIOS = RAIZ / "models" / f"bertimbau-pii-v1-cenarios{_sufixo_experimento}"

TIPOS = ["CPF", "RG", "NOME", "ENDERECO"]
LABEL_LIST = ["O"] + [f"{prefixo}-{tipo}" for tipo in TIPOS for prefixo in ("B", "I")]
ID2LABEL = {i: l for i, l in enumerate(LABEL_LIST)}

resultados_cenarios = json.loads((PASTA_CENARIOS / "_comparacao_cenarios.json").read_text(encoding="utf-8"))
cenario_escolhido = next((r for r in resultados_cenarios if r["nome"] == CENARIO_MANUAL), resultados_cenarios[0]) if CENARIO_MANUAL else resultados_cenarios[0]

print(f"Experimento: {PASTA_CENARIOS.name}")
print(f"Cenário: {cenario_escolhido['nome']} (F1 token-level={cenario_escolhido['eval_f1']:.4f})")
print(f"Pasta do modelo: {cenario_escolhido['pasta_modelo']}")

arquivos_extraidos = sorted(PASTA_EXTRACAO_HOLDOUT.glob("*.json"))
print(f"\nDocumentos extraídos no holdout: {len(arquivos_extraidos)}")
if not arquivos_extraidos:
    print(f"[AVISO] Nenhum JSON em {PASTA_EXTRACAO_HOLDOUT} -- rode o notebook 08 primeiro.")

Experimento: bertimbau-pii-v1-cenarios-sem-sinteticos
Cenário: c08_warmup (F1 token-level=0.8243)
Pasta do modelo: d:\Arquivos\ProjetosPython\BFC\LocalScanDocument_v3\models\bertimbau-pii-v1-cenarios-sem-sinteticos\c08_warmup

Documentos extraídos no holdout: 30


## Carrega o modelo

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(cenario_escolhido["pasta_modelo"])
model = AutoModelForTokenClassification.from_pretrained(cenario_escolhido["pasta_modelo"])
model.to("cuda" if torch.cuda.is_available() else "cpu")
model.eval()
print(f"Modelo carregado: {cenario_escolhido['nome']}")


from pos_processamento_ner import mesclar_previsoes_por_janela


@torch.no_grad()
def _prever_tags_janela(modelo, tokens_janela: list) -> list:
    encoding = tokenizer(tokens_janela, is_split_into_words=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
    encoding = {k: v.to(modelo.device) for k, v in encoding.items()}
    logits = modelo(**encoding).logits[0]
    pred_ids = logits.argmax(dim=-1).tolist()
    word_ids = tokenizer(tokens_janela, is_split_into_words=True, truncation=True, max_length=MAX_LENGTH).word_ids()
    tags_por_palavra = ["O"] * len(tokens_janela)
    visto = set()
    for pid, word_idx in zip(pred_ids, word_ids):
        if word_idx is None or word_idx in visto:
            continue
        visto.add(word_idx)
        tags_por_palavra[word_idx] = ID2LABEL[pid]
    return tags_por_palavra


def prever_pagina(modelo, tokens: list) -> list:
    """Prevê a página inteira, dividindo em janelas deslizantes com sobreposição quando
    ultrapassa MAX_LENGTH subtokens (achado 2026-09-15, caso MINAS CAL/segundo endereço)."""
    return mesclar_previsoes_por_janela(tokens, tokenizer, lambda tj: _prever_tags_janela(modelo, tj), max_length=MAX_LENGTH)


def agrupar_bboxes_por_linha(tokens_bbox: list, indices: list) -> list:
    if not indices:
        return []
    grupos = []
    grupo_atual = [tokens_bbox[indices[0]]]
    y_ref = tokens_bbox[indices[0]][1]
    for i in indices[1:]:
        bbox = tokens_bbox[i]
        altura_ref = grupo_atual[-1][3] - grupo_atual[-1][1]
        if abs(bbox[1] - y_ref) > max(altura_ref * 0.6, 3):
            grupos.append(grupo_atual)
            grupo_atual = [bbox]
            y_ref = bbox[1]
        else:
            grupo_atual.append(bbox)
    grupos.append(grupo_atual)
    return [[min(b[0] for b in g), min(b[1] for b in g), max(b[2] for b in g), max(b[3] for b in g)] for g in grupos]


def decodificar_bio(tags_previstas: list, tokens_bbox: list, tokens: list) -> list:
    entidades = []
    i = 0
    n = len(tags_previstas)
    while i < n:
        tag = tags_previstas[i]
        if tag == "O":
            i += 1
            continue
        tipo = tag.split("-", 1)[1]
        indices = [i]
        j = i + 1
        while j < n and tags_previstas[j] == f"I-{tipo}":
            indices.append(j)
            j += 1
        entidades.append({
            "type": tipo,
            "bbox": agrupar_bboxes_por_linha(tokens_bbox, indices),
            "indices": indices,
            "text": " ".join(tokens[k] for k in indices),
        })
        i = j
    return entidades


def bbox_externo(lista_bboxes: list):
    if not lista_bboxes:
        return None
    return [min(b[0] for b in lista_bboxes), min(b[1] for b in lista_bboxes),
            max(b[2] for b in lista_bboxes), max(b[3] for b in lista_bboxes)]


def iou(a, b) -> float:
    if a is None or b is None:
        return 0.0
    xi0, yi0 = max(a[0], b[0]), max(a[1], b[1])
    xi1, yi1 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, xi1 - xi0) * max(0, yi1 - yi0)
    if inter == 0:
        return 0.0
    area_a = max(0, a[2] - a[0]) * max(0, a[3] - a[1])
    area_b = max(0, b[2] - b[0]) * max(0, b[3] - b[1])
    uniao = area_a + area_b - inter
    return inter / uniao if uniao > 0 else 0.0

## Roda a inferência em todos os documentos do holdout e lista as entidades encontradas

In [4]:
resultados_por_documento = {}

for arq in arquivos_extraidos:
    extracao = json.loads(arq.read_text(encoding="utf-8"))
    doc_id = extracao["document_id"]
    entidades_documento = []

    for pagina_dados in extracao["pages"]:
        tokens_dict = pagina_dados["tokens"]
        if not tokens_dict:
            continue
        tokens = [t["text"] for t in tokens_dict]
        tokens_bbox = [t["bbox"] for t in tokens_dict]

        tags_previstas = prever_pagina(model, tokens)
        previstas = decodificar_bio(tags_previstas, tokens_bbox, tokens)

        if APLICAR_REGEX_ASSINATURA_DIGITAL:
            extras = extrair_nome_assinatura_digital(tokens, tokens_bbox)
            for extra in extras:
                bbox_extra = bbox_externo(extra["bbox"])
                ja_coberto = any(
                    p["type"] == "NOME" and iou(bbox_extra, bbox_externo(p["bbox"])) > 0.1
                    for p in previstas
                )
                if not ja_coberto:
                    previstas.append(extra)

        for p in previstas:
            p["page"] = pagina_dados["page"]
        entidades_documento.extend(previstas)

    if APLICAR_PROPAGACAO_TEXTO_IDENTICO:
        # `propagar_entidades_por_texto_identico` agora devolve a lista COMPLETA já corrigida
        # (substitui, não concatena) -- mudança de contrato 2026-09-15: quando uma entidade
        # existente cobre só uma PARTE do span canônico repetido em outra página (ex.: modelo
        # previu só "302" onde o texto completo do carimbo é "AV. BIAS FORTES, 932 CONJ. 302
        # [...]"), ela é substituída pelo span completo, priorizando consistência de marcação
        # entre páginas idênticas do mesmo documento.
        entidades_documento = propagar_entidades_por_texto_identico(
            entidades_documento, extracao["pages"], tipos_propagaveis=tuple(TIPOS)
        )

    resultados_por_documento[doc_id] = entidades_documento

    print(f"\n{'=' * 90}\n{doc_id}\n{'=' * 90}")
    if not entidades_documento:
        print("  (nenhuma entidade encontrada)")
    for e in entidades_documento:
        origem = f" [{e['origem']}]" if "origem" in e else ""
        print(f"  p{e['page']:<3} {e['type']:<9} \"{e['text']}\"{origem}")

n_total = sum(len(v) for v in resultados_por_documento.values())
print(f"\n\nTotal: {len(resultados_por_documento)} documento(s), {n_total} entidade(s) encontrada(s).")


cm1iasv3k0031dhqdfn4663iw-Alteração 11 - MINAS CAL.pdf
  p1   NOME      "Marinely de Paula Bomfim" [propagado_texto_identico]
  p1   NOME      "Marinely de Paula Bomfim" [propagado_texto_identico]
  p2   CPF       "274.228.956-91"
  p2   NOME      "SERGIO LUCIO SOARES"
  p2   NOME      "Marinely de Paula Bomfim"
  p2   NOME      "Marinely de Paula Bomfim" [propagado_texto_identico]
  p3   ENDERECO  "BIAS FORTES, 932 CONJ. 302 – LOURDES – BELO HORIZONTE/MG – CEP: 30.170-013."
  p3   NOME      "ELIZABETH APARECIDA SOARES,"
  p3   RG        "M-2.129.482"
  p3   CPF       "721.516.416-00,"
  p3   ENDERECO  "rua França, nº 102, bairro Nações Unidas, Sabará/MG, CEP: 34.590-110;"
  p3   CPF       "37.089.378-5"
  p3   CPF       "274.228.956-91," [propagado_texto_identico]
  p3   NOME      "SÉRGIO LÚCIO SOARES," [propagado_texto_identico]
  p3   NOME      "Marinely de Paula Bomfim" [propagado_texto_identico]
  p3   NOME      "Marinely de Paula Bomfim" [propagado_texto_identico]
  p4   NOME   

## Visualização: todas as páginas do PDF com as caixas previstas

Preencha `DOC_ID_VISUALIZAR` (use um `document_id` impresso acima) para ver as caixas desenhadas sobre **todas** as páginas do documento, uma figura por página.

In [ ]:
import fitz  # PyMuPDF
from io import BytesIO
from PIL import Image

DOC_ID_VISUALIZAR = next(iter(resultados_por_documento), None)  # troque para outro document_id da lista acima


def localizar_pdf_original(document_id: str) -> Path:
    candidato = PASTA_ARQUIVOS_HOLDOUT / document_id
    if candidato.exists():
        return candidato
    achados = list(PASTA_ARQUIVOS_HOLDOUT.glob(f"{Path(document_id).stem}*"))
    if achados:
        return achados[0]
    raise FileNotFoundError(f"PDF original não encontrado para {document_id} em {PASTA_ARQUIVOS_HOLDOUT}")


CORES_POR_TIPO = {
    "CPF": (0.85, 0.1, 0.1),
    "RG": (0.1, 0.4, 0.85),
    "NOME": (0.1, 0.7, 0.2),
    "ENDERECO": (0.8, 0.6, 0.0),
}

if DOC_ID_VISUALIZAR:
    caminho_pdf = localizar_pdf_original(DOC_ID_VISUALIZAR)
    doc = fitz.open(caminho_pdf)
    entidades_documento = resultados_por_documento[DOC_ID_VISUALIZAR]
    legenda = "  ".join(f"{t}={c}" for t, c in CORES_POR_TIPO.items())

    for pagina_pdf in doc:
        numero_pagina = pagina_pdf.number + 1
        entidades_pagina = [e for e in entidades_documento if e["page"] == numero_pagina]
        for e in entidades_pagina:
            # Um retângulo por linha de e["bbox"] -- ver nota em notebooks/bert/10 (achado
            # 2026-09-15): bbox_externo() sozinho "vaza" para a esquerda em entidades que
            # quebram linha.
            for retangulo in (e.get("bbox") or []):
                pagina_pdf.draw_rect(fitz.Rect(*retangulo), color=CORES_POR_TIPO.get(e["type"], (0, 0, 0)), width=1.5)

        pix = pagina_pdf.get_pixmap(dpi=150)
        img_bytes = pix.tobytes("png")

        plt.figure(figsize=(11, 15))
        plt.imshow(Image.open(BytesIO(img_bytes)))
        plt.axis("off")
        plt.title(f"{DOC_ID_VISUALIZAR} — página {numero_pagina}/{doc.page_count} ({len(entidades_pagina)} entidade(s))\n{legenda}")
        plt.tight_layout()
        plt.show()

    doc.close()
else:
    print("Nenhum documento no holdout -- rode o notebook 08 primeiro.")